# AFS-DSN: Evaluation

This notebook handles all evaluation tasks:

1. **Test set evaluation** — Dice, HD95, ASD, thin-wall Dice (NasalSeg n=20)
2. **Computational cost comparison** — Params, FLOPs, GPU memory, inference time
3. **External validation** — Braz J (n=20) and CRSwNP (n=81) zero-shot inference
4. **Baseline comparisons** — Evaluate all baselines with same setup

**Requires:** `02_model.ipynb` and trained checkpoints from `03_train.ipynb`

In [1]:
# ============================================================
# Cell 1: CONFIG
# ============================================================

DATA_ROOT       = '/workspace/zenodo_tmp'           # NasalSeg
CHECKPOINT_FULL = './checkpoints_full/Stage4_FullModel_best.pth'
CHECKPOINT_LITE = './checkpoints_lite/Stage4_FullModel_best.pth'
SPLIT_JSON      = './checkpoints_full/split_indices.json'  # must match training
OUTPUT_DIR      = './results'

# External dataset paths (set to None if not available)
BRAZ_J_ROOT  = None   # e.g. '/workspace/braz_j'
CRSWP_ROOT   = None   # e.g. '/workspace/crswp'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready')

Config ready


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 
                       'numpy==1.26.4', 'scipy==1.11.4', '--quiet'])
print('✅ Fixed. Now restart kernel and run again.')


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


✅ Fixed. Now restart kernel and run again.


In [3]:
# ============================================================
# Cell 2: Imports
# ============================================================
%run 02_model.ipynb

compute_dice = compute_dice_np

from scipy.ndimage import distance_transform_edt
from skimage.segmentation import find_boundaries
import json
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import matplotlib.pyplot as plt
def load_checkpoint(path, variant='full'):
    ck = torch.load(path, map_location=device)
    model = AFS_DSN_V4() if variant == 'full' else AFS_DSN_Lite()
    model = model.to(device)
    model.load_state_dict(ck['model'])
    print(f'✅ Loaded: {path}')
    print(f'   Epoch: {ck["epoch"]}  Best Dice: {ck["best_dice"]:.4f}')
    return model, ck


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip i

✅ Packages ready
Device: cuda
GPU: NVIDIA A40
VRAM: 47.7 GB
✅ MultiScaleWavelet3D, DoubleConv
✅ FrequencyBranchV4 (Full)
✅ FrequencyBranchLite
✅ CrossDomainAttention, AdaptiveRouter
AFS_DSN_V4 Full: 414.57M params
AFS_DSN_Lite: 27.41M params
✅ AFS_DSN_V4, AFS_DSN_Lite
✅ NasalSegDataset
✅ CombinedLoss, dice_coefficient, compute_dice_np
02_model.ipynb ready
  AFS_DSN_V4()   — full (~414M)
  AFS_DSN_Lite() — lite (~80M)
  NasalSegDataset(root)
  CombinedLoss() — CE + Dice


In [4]:
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import find_boundaries
import numpy as np

compute_dice = compute_dice_np

def compute_hd95(pred, target, spacing=(1.0,1.0,1.0)):
    if pred.sum()==0 or target.sum()==0:
        return float('nan')
    dist_pred   = distance_transform_edt(~pred.astype(bool),   sampling=spacing)
    dist_target = distance_transform_edt(~target.astype(bool), sampling=spacing)
    surf_pred   = find_boundaries(pred,   mode='outer')
    surf_target = find_boundaries(target, mode='outer')
    d1 = dist_target[surf_pred]
    d2 = dist_pred[surf_target]
    if len(d1)==0 or len(d2)==0:
        return float('nan')
    return float(np.percentile(np.concatenate([d1, d2]), 95))

def compute_asd(pred, target, spacing=(1.0,1.0,1.0)):
    if pred.sum()==0 or target.sum()==0:
        return float('nan')
    dist_pred   = distance_transform_edt(~pred.astype(bool),   sampling=spacing)
    dist_target = distance_transform_edt(~target.astype(bool), sampling=spacing)
    surf_pred   = find_boundaries(pred,   mode='outer')
    surf_target = find_boundaries(target, mode='outer')
    d1 = dist_target[surf_pred]
    d2 = dist_pred[surf_target]
    if len(d1)==0 or len(d2)==0:
        return float('nan')
    return float((d1.mean() + d2.mean()) / 2)

def compute_thin_wall_dice(pred, target, boundary_mm=2.0, spacing=(1.0,1.0,1.0)):
    dist = distance_transform_edt(~target.astype(bool), sampling=spacing)
    mask = dist < boundary_mm
    if mask.sum() == 0:
        return float('nan')
    return compute_dice(pred[mask], target[mask])

print('✅ All metrics defined')

✅ All metrics defined


In [5]:
# ============================================================
# Cell 3: Load test split
# ============================================================
with open(SPLIT_JSON) as f:
    split = json.load(f)

full_dataset = NasalSegDataset(DATA_ROOT)
test_dataset = Subset(full_dataset, split['test'])
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f'Test set: {len(test_dataset)} cases')
print(f'Split seed: {split["seed"]}')

📊 Found 130 samples
Test set: 20 cases
Split seed: 42


In [6]:
# ============================================================
# Cell 4: Core evaluation function
# ============================================================

@torch.no_grad()
def evaluate_model(model, loader, desc='Evaluating'):
    """
    Returns dict with per-case and mean metrics:
    dice, hd95, asd, thin_wall_dice, thick_wall_dice
    """
    model.eval()
    rows = []
    for i, (images, masks) in enumerate(tqdm(loader, desc=desc)):
        images = images.to(device)
        out  = model(images)['output']
        pred = out.argmax(dim=1).cpu().numpy()[0].astype(np.uint8)  # (D,H,W)
        gt   = masks.numpy()[0].astype(np.uint8)                    # (D,H,W)

        dice      = compute_dice(pred, gt)
        hd95      = compute_hd95(pred, gt)
        asd       = compute_asd(pred, gt)
        tw_dice   = compute_thin_wall_dice(pred, gt, boundary_mm=2.0)

        # Thick-wall: foreground voxels far from boundary (≥2mm)
        from scipy.ndimage import distance_transform_edt
        # 距離是從 foreground 的邊界往內量
        dist_inside = distance_transform_edt(gt.astype(bool))  # ← 改這裡，拿掉 ~
        thick_mask = (dist_inside >= 2.0)
        if thick_mask.sum() > 0:
            thk_dice = compute_dice(pred[thick_mask], gt[thick_mask])
        else:
            thk_dice = float('nan')

        rows.append({
            'case': i,
            'dice': dice,
            'hd95': hd95,
            'asd':  asd,
            'thin_wall_dice':  tw_dice,
            'thick_wall_dice': thk_dice,
        })

    df = pd.DataFrame(rows)
    print(f'\n  Overall Dice:          {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
    print(f'  HD95 (mm):             {df["hd95"].mean():.3f}')
    print(f'  ASD  (mm):             {df["asd"].mean():.3f}')
    print(f'  Thin-wall Dice (<2mm): {df["thin_wall_dice"].mean():.4f} ± {df["thin_wall_dice"].std():.4f}')
    print(f'  Thick-wall Dice (≥2mm):{df["thick_wall_dice"].mean():.4f} ± {df["thick_wall_dice"].std():.4f}')
    return df

print('✅ evaluate_model ready')

✅ evaluate_model ready


In [7]:
# ============================================================
# Cell 5: Evaluate Full model on test set
# ============================================================
model_full, _ = load_checkpoint(CHECKPOINT_FULL, variant='full')
print('\n--- AFS-DSN Full (test set n=20) ---')
df_full = evaluate_model(model_full, test_loader, desc='Full model')
df_full.to_csv(f'{OUTPUT_DIR}/test_results_full.csv', index=False)

/tmp/ipykernel_372/1273011437.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck = torch.load(path, map_location=device)


✅ Loaded: ./checkpoints_full/Stage4_FullModel_best.pth
   Epoch: 16  Best Dice: 0.9574

--- AFS-DSN Full (test set n=20) ---


Full model: 100%|██████████| 20/20 [00:48<00:00,  2.44s/it]


  Overall Dice:          0.9423 ± 0.0221
  HD95 (mm):             1.862
  ASD  (mm):             0.992
  Thin-wall Dice (<2mm): 0.9457 ± 0.0198
  Thick-wall Dice (≥2mm):0.9985 ± 0.0013


In [8]:
# ============================================================
# Cell 6: Evaluate Lite model on test set
# ============================================================
model_lite, _ = load_checkpoint(CHECKPOINT_LITE, variant='lite')
print('\n--- AFS-DSN Lite (test set n=20) ---')
df_lite = evaluate_model(model_lite, test_loader, desc='Lite model')
df_lite.to_csv(f'{OUTPUT_DIR}/test_results_lite.csv', index=False)

/tmp/ipykernel_372/1273011437.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck = torch.load(path, map_location=device)


✅ Loaded: ./checkpoints_lite/Stage4_FullModel_best.pth
   Epoch: 20  Best Dice: 0.9591

--- AFS-DSN Lite (test set n=20) ---


Lite model: 100%|██████████| 20/20 [00:49<00:00,  2.47s/it]


  Overall Dice:          0.9437 ± 0.0234
  HD95 (mm):             1.821
  ASD  (mm):             0.989
  Thin-wall Dice (<2mm): 0.9467 ± 0.0212
  Thick-wall Dice (≥2mm):0.9983 ± 0.0015


In [9]:
# ============================================================
# Cell 7: Computational cost comparison (Table 2 in paper)
# ============================================================
try:
    from fvcore.nn import FlopCountAnalysis
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fvcore', '-q'])
    from fvcore.nn import FlopCountAnalysis

def measure_cost(model, label, n_runs=10):
    model.eval()
    x = torch.randn(1, 1, 128, 128, 128).to(device)
    total = count_params(model)

    # FLOPs
    flops = FlopCountAnalysis(model, x).total() / 1e9

    # GPU memory
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(x)
    mem_gb = torch.cuda.max_memory_allocated() / 1e9

    # Inference time
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            torch.cuda.synchronize()
            t0 = time.time()
            _ = model(x)
            torch.cuda.synchronize()
            times.append((time.time() - t0) * 1000)
    inf_ms = float(np.mean(times[2:]))  # skip warm-up

    return {
        'Model': label,
        'Params (M)': round(total / 1e6, 2),
        'FLOPs (G)':  round(flops, 2),
        'GPU Mem (GB)': round(mem_gb, 2),
        'Inference (ms)': round(inf_ms, 1),
    }

results = []
results.append(measure_cost(model_full, 'AFS-DSN Full'))
results.append(measure_cost(model_lite, 'AFS-DSN Lite'))

df_cost = pd.DataFrame(results)
print(df_cost.to_string(index=False))
df_cost.to_csv(f'{OUTPUT_DIR}/computational_cost.csv', index=False)


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Unsupported operator aten::leaky_relu_ encountered 24 time(s)
Unsupported operator aten::add encountered 12 time(s)
Unsupported operator aten::max_pool3d encountered 4 time(s)
Unsupported operator aten::lift_fresh encountered 1536 time(s)
Unsupported operator aten::mul encountered 91 time(s)
Unsupported operator aten::leaky_relu encountered 24 time(s)
Unsupported operator aten::upsample_trilinear3d encountered 24 time(s)
Unsupported operator aten::abs encountered 8 time(s)
Unsupported operator aten::mean encountered 9 time(s)
Unsupported operator aten::softmax encountered 3 time(s)
Unsupported operator aten::std encountered 1 time(s)
Unsupported operator aten::leaky_relu_ encountered 23 time(s)
Unsupported operator aten::add encountered 14 time(s)
Unsupported operator aten::max_pool3d encountered 4 time(s)
Unsupported operator aten::lift_fresh encountered 1536 time(s

       Model  Params (M)  FLOPs (G)  GPU Mem (GB)  Inference (ms)
AFS-DSN Full      414.57     686.29          4.18           278.8
AFS-DSN Lite       27.41     488.05          4.18           271.1


In [12]:
# ============================================================
# Cell 8: External validation — zero-shot inference
# (Braz J / CRSwNP — run after pointing BRAZ_J_ROOT / CRSWP_ROOT)
# ============================================================

def external_validate(model, data_root, dataset_name, model_label):
    """Zero-shot inference on external dataset."""
    if data_root is None:
        print(f'⚠️  {dataset_name}: data_root is None, skipping.')
        return None
    try:
        ext_dataset = NasalSegDataset(data_root)
    except FileNotFoundError as e:
        print(f'⚠️  {dataset_name}: {e}')
        return None
    ext_loader = DataLoader(ext_dataset, batch_size=1, shuffle=False, num_workers=2)
    print(f'\n--- {dataset_name} ({len(ext_dataset)} cases) — {model_label} ---')
    df = evaluate_model(model, ext_loader, desc=dataset_name)
    df['dataset'] = dataset_name
    df['model']   = model_label
    df.to_csv(f'{OUTPUT_DIR}/external_{dataset_name.lower().replace(" ","_")}_{model_label.lower()}.csv',
              index=False)
    return df

# Run on both external datasets with Full model
df_braz  = external_validate(model_full, BRAZ_J_ROOT,  'Braz_J',  'Full')
df_crswp = external_validate(model_full, CRSWP_ROOT,   'CRSwNP',  'Full')

# Optionally also run Lite model on external sets
# df_braz_lite = external_validate(model_lite, BRAZ_J_ROOT, 'Braz_J', 'Lite')

⚠️  Braz_J: data_root is None, skipping.
⚠️  CRSwNP: data_root is None, skipping.


In [13]:
# ============================================================
# Cell 9: Summary table (ready for paper)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

def summarise(df, label):
    return {
        'Model':           label,
        'Dice (%)':        f"{df['dice'].mean()*100:.2f} ± {df['dice'].std()*100:.2f}",
        'HD95 (mm)':       f"{df['hd95'].mean():.2f}",
        'ASD (mm)':        f"{df['asd'].mean():.3f}",
        'TW Dice (%)':     f"{df['thin_wall_dice'].mean()*100:.2f} ± {df['thin_wall_dice'].std()*100:.2f}",
    }

rows = [summarise(df_full, 'AFS-DSN Full  (n=20)'),
        summarise(df_lite, 'AFS-DSN Lite  (n=20)')]

if df_braz is not None:
    rows.append(summarise(df_braz,  'Full → Braz J (n=20, zero-shot)'))
if df_crswp is not None:
    rows.append(summarise(df_crswp, 'Full → CRSwNP (n=81, zero-shot)'))

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv(f'{OUTPUT_DIR}/summary_table.csv', index=False)
print(f'\nSaved: {OUTPUT_DIR}/summary_table.csv')

               Model     Dice (%) HD95 (mm) ASD (mm)  TW Dice (%)
AFS-DSN Full  (n=20) 94.23 ± 2.21      1.86    0.992 94.57 ± 1.98
AFS-DSN Lite  (n=20) 94.37 ± 2.34      1.82    0.989 94.67 ± 2.12

Saved: ./results/summary_table.csv
